# Testes das classes base

Notebook pra validar `AgentRole`, `AgentContext`, `AgentResponse`, `BaseAgent` e `AgenteInvestimentos`, usando um LLM real da OpenAI (`ChatOpenAI`) via LangChain. Requer um `.env` com `OPENAI_API_KEY` preenchida.

In [12]:
import importlib
import sys

# Remove do cache
if 'src.agents.dividas' in sys.modules:
    del sys.modules['src.agents.dividas']

# Reimporta o módulo atualizado
from src.agents.dividas import AgenteDividas

print("Módulo recarregado com sucesso!")

Módulo recarregado com sucesso!


## 0. Setup — deixar o projeto importável

In [1]:
import sys
from pathlib import Path

# Sobe um nível (de notebooks/ para a raiz do projeto) e adiciona ao path,
# assim `from src...` funciona igual funcionaria rodando a partir da raiz.
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Raiz do projeto: {project_root}")

Raiz do projeto: c:\2_Estudos\conselheiro-financeiro-ia


## 1. `AgentRole` — testando o Enum

In [2]:
from src.core.schemas import AgentRole

print(AgentRole.INVESTIMENTOS)
print(AgentRole.INVESTIMENTOS.value)

# Comparação direta com string funciona por causa do (str, Enum)
print(AgentRole.INVESTIMENTOS == "investimentos")

# Converter uma string vinda de fora (ex: JSON) para o Enum
print(AgentRole("dividas"))

# String inválida deve dar erro -- é a proteção contra typo
try:
    AgentRole("investimento")  # sem o 's' -- typo proposital
except ValueError as e:
    print(f"Erro esperado: {e}")

AgentRole.INVESTIMENTOS
investimentos
True
AgentRole.DIVIDAS
Erro esperado: 'investimento' is not a valid AgentRole


## 2. `AgentContext` — testando o schema de entrada

In [3]:
from src.core.schemas import AgentContext

contexto = AgentContext(
    session_id="sessao-teste-001",
    user_message="O que é tesouro direto?",
)

print(contexto)
print()
# history e metadata vieram vazios por padrão -- confirma que default_factory funcionou
print(f"history: {contexto.history}")
print(f"metadata: {contexto.metadata}")

session_id='sessao-teste-001' user_message='O que é tesouro direto?' history=[] metadata={}

history: []
metadata: {}


In [ ]:
# Validação de tipo do Pydantic em ação: isso deve dar erro,
# porque session_id precisa ser string
from pydantic import ValidationError

try:
    AgentContext(session_id=123, user_message="teste")
except ValidationError as e:
    print(f"Erro esperado de validação:\n{e}")

## 3. `BaseAgent` — confirmando que não dá pra instanciar direto

In [6]:
from src.agents.base import BaseAgent

try:
    BaseAgent(llm=None)
except TypeError as e:
    print(f"Erro esperado (classe abstrata): {e}")

Erro esperado (classe abstrata): Can't instantiate abstract class BaseAgent with abstract method run


## 4. `AgenteInvestimentos` com a OpenAI de verdade

Carregamos a chave do `.env` e instanciamos um `ChatOpenAI` real. Repare que o `AgenteInvestimentos` não muda em nada -- é a mesma classe, só o objeto `llm` injetado no construtor é que é real agora em vez de falso (injeção de dependência).

In [ ]:
import os

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(project_root / ".env") # carrega as variáveis de ambiente do arquivo .env na raiz do projeto

assert os.environ.get("OPENAI_API_KEY"), (
    "OPENAI_API_KEY não encontrada -- confira se o .env existe na raiz "
    "do projeto e está preenchido."
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

In [8]:
from src.agents.investimentos import AgenteInvestimentos
from src.core.schemas import AgentContext

agente = AgenteInvestimentos(llm=llm)

contexto = AgentContext(
    session_id="sessao-teste-001",
    user_message="O que é tesouro direto?",
)

resposta = agente.run(contexto)

print()
print(f"agent: {resposta.agent}")
print(f"content: {resposta.content}")
print(f"requires_review: {resposta.requires_review}")
print(f"created_at: {resposta.created_at}")


agent: AgentRole.INVESTIMENTOS
content: O Tesouro Direto é um programa do governo brasileiro que permite que pessoas físicas comprem títulos públicos, ou seja, dívidas emitidas pelo governo para financiar suas atividades e projetos. Ao investir no Tesouro Direto, você está basicamente emprestando dinheiro ao governo em troca de uma promessa de pagamento de juros e devolução do valor investido após um determinado período.

Aqui estão alguns pontos importantes sobre o Tesouro Direto:

1. **Tipos de Títulos**: Existem diferentes tipos de títulos disponíveis, que podem ser classificados em:
   - **Tesouro Selic**: Títulos que acompanham a taxa Selic, ideal para quem busca segurança e liquidez.
   - **Tesouro Prefixado**: Títulos com uma taxa de juros fixa, que garante um retorno conhecido no momento da compra.
   - **Tesouro IPCA+**: Títulos que oferecem uma rentabilidade atrelada à inflação (medida pelo IPCA) mais uma taxa de juros fixa, protegendo o poder de compra do investidor.

2. **

## 5. Agente de Dívidas com a OpenAI

In [13]:
import os

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(project_root / ".env") # carrega as variáveis de ambiente do arquivo .env na raiz do projeto

assert os.environ.get("OPENAI_API_KEY"), (
    "OPENAI_API_KEY não encontrada -- confira se o .env existe na raiz "
    "do projeto e está preenchido."
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

In [18]:
from src.agents.dividas import AgenteDividas
from src.core.schemas import AgentContext

agente = AgenteDividas(llm=llm)

contexto = AgentContext(
    session_id="sessao-teste-003",
    user_message="Estou muito endividado com a fatura do cartão de crédito e empréstimos. Corro o risco de perder minha casa. O que faço?",
)

resposta = agente.run(contexto)

print(f"System Prompt: {agente.system_prompt}")
print(f"agent: {resposta.agent}")
print(f"content: {resposta.content}")
print(f"requires_review: {resposta.requires_review}")
print(f"created_at: {resposta.created_at}")
#print(f"User Message: {resposta.user_message}")

System Prompt: Você é um educador financeiro. Sua função é explicar conceitos de investimentos relacionados a dívidas de forma didática e acessível para ajudar pessoas endividas a saírem dessa situação
e conseguirem começar a guardar dinheiro para reserva de emergência, investimentos e sonhos.

REGRA ABSOLUTA: nunca recomende um ativo, ticker, fundo, correta ou instituição financeira específica. 
Seu papel é dar conhecimento para que a pessoa decida por conta própria, nunca decidir por ela

agent: AgentRole.DIVIDAS
content: Sinto muito em saber que você está passando por essa situação difícil. Vamos abordar isso de forma prática e didática, focando em algumas etapas que podem ajudá-lo a sair dessa situação.

### 1. **Avalie sua Situação Financeira**
   - **Liste suas Dívidas**: Anote todas as suas dívidas, incluindo o valor total, a taxa de juros e a data de vencimento. Isso ajuda a ter uma visão clara do que você deve.
   - **Renda Mensal**: Calcule sua renda mensal líquida (o que voc

## 5. Testando com histórico de conversa

Confirma que `_build_messages` está juntando system prompt + histórico + mensagem atual corretamente.

In [19]:
contexto_com_historico = AgentContext(
    session_id="sessao-teste-001",
    user_message="E fundos imobiliários, o que são?",
    history=[
        {"role": "user", "content": "O que é tesouro direto?"},
        {"role": "assistant", "content": "É uma plataforma para comprar títulos públicos."},
    ],
)

resposta2 = agente.run(contexto_com_historico)
print()
print(f"content: {resposta2.content}")


content: Fundos imobiliários, ou FIIs, são uma forma de investimento coletivo em imóveis. Quando você investe em um fundo imobiliário, está comprando cotas de um fundo que, por sua vez, adquire e administra propriedades, como edifícios comerciais, shopping centers, hospitais, entre outros.

Aqui estão alguns pontos importantes sobre os fundos imobiliários:

1. **Renda Passiva**: Os FIIs geralmente distribuem rendimentos mensais aos cotistas, que vêm da locação dos imóveis ou da venda de propriedades. Isso pode ser uma forma de gerar uma renda passiva.

2. **Diversificação**: Ao investir em um fundo, você pode ter acesso a uma carteira diversificada de imóveis, o que pode reduzir o risco em comparação a investir em um único imóvel.

3. **Liquidez**: As cotas dos fundos imobiliários são negociadas na bolsa de valores, o que significa que você pode comprar e vender suas cotas com mais facilidade do que vender um imóvel físico.

4. **Gestão Profissional**: Os fundos são geridos por profis

## Próximos passos

- Se quiser trocar de provedor no futuro (Anthropic ou Groq), troque só a célula que instancia `llm` por `ChatAnthropic(model="...")` ou `ChatGroq(model="...")` -- nada no `AgenteInvestimentos` muda, é exatamente esse o ganho da injeção de dependência.
- Fique de olho no consumo de créditos da OpenAI ao rodar este notebook várias vezes -- cada execução da seção 4 e 5 é uma chamada real de API.